# L14c: Multi-Input Multi-Output State Space Models and Forecasting
This lecture extends the [L14a SISO structured state space model](../L14a) along two axes. First, we replace scalar inputs and scalar outputs with vector inputs $\mathbf{u}_{t}\in\mathbb{R}^{d_{\text{in}}}$ and vector outputs $\mathbf{y}_{t}\in\mathbb{R}^{d_{\text{out}}}$, yielding a multi-input multi-output (MIMO) SSM. Second, we replace the memorize target with a genuine _forecasting_ target, $\mathbf{y}_{t} = \mathbf{u}_{t+k}$, which turns the same ridge-regression machinery into a predictor rather than a reconstructor.

We stay in the S4 tradition, so the MIMO state matrix $\mathbf{A}$ is _block-diagonal_: we run one independent LegS filter per input channel and concatenate their hidden states. The readout matrix $\mathbf{C}$ is dense, so every output channel can mix information from every input channel's history. Because $(\bar{\mathbf{A}}, \bar{\mathbf{B}})$ are frozen by the HiPPO theory, the training loss remains quadratic in $\bar{\mathbf{C}}$ and $\bar{\mathbf{D}}$, and we can still fit them by closed-form ridge regression, now as a stacked multi-output solve.

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Write down a MIMO SSM and identify the shape of each of its four matrices:__ Given a choice of hidden dimension per channel and of input and output channel counts, state the sizes of the state, input, readout, and feedforward matrices. Describe the role each matrix plays in the vector recursion.
> * __Explain the block-diagonal MIMO LegS construction and why it reduces to per-channel SISO filters:__ Describe how running one LegS filter per input channel yields a block-diagonal state matrix and a block-rectangular input matrix. Explain why this preserves the stability and approximation guarantees of the single-channel construction.
> * __Frame a forecasting task as a stacked ridge regression on hidden states:__ Convert an input sequence and a forecast horizon into an aligned input-target pair, and assemble the hidden-state matrix via a single rollout of the shared filter. Solve one multi-output ridge system for every output channel simultaneously.

Let's get started!
___

## Example
Today, we will use the following notebook to illustrate key concepts:

> [▶ Two-channel oscillator forecasting with a MIMO HiPPO-LegS SSM](CHEME-5820-L14c-Example-TwoChannelOscillator-Spring-2026.ipynb). In this example, we generate a two-channel synthetic signal from a pair of sinusoids at different frequencies, fit a MIMO LegS SSM to forecast the signal one step ahead, and evaluate the forecast on a held-out continuation of the series. We sweep the hidden dimension $h$ and the forecast horizon $k$ to see how memory and predictive lead time interact.

___

<div>
    <center>
      <img
        src="figs/Fig-MIMO-SSM-Block.svg"
        alt="Discrete-time MIMO SSM block diagram"
        height="360"
        width="900"
      />
    </center>
</div>

## From SISO to MIMO
In [L14a](../L14a), the SISO state space model mapped a scalar input $u_{t}\in\mathbb{R}$ to a scalar output $y_{t}\in\mathbb{R}$ through an $h$-dimensional hidden state. In the multi-input multi-output version, the input and output both become vectors and the hidden state grows accordingly.

> __Continuous-Time MIMO SSM__
>
> Let $\mathbf{u}:[0, T]\to\mathbb{R}^{d_{\text{in}}}$ be a vector-valued input and let $\mathbf{x}:[0, T]\to\mathbb{R}^{H}$ be a hidden-state trajectory, where $H$ denotes the total hidden-state dimension. The continuous-time MIMO SSM is
> $$
\boxed{
\begin{align*}
\dot{\mathbf{x}}(t) &= \mathbf{A}\,\mathbf{x}(t) + \mathbf{B}\,\mathbf{u}(t) \\
\mathbf{y}(t)      &= \mathbf{C}\,\mathbf{x}(t) + \mathbf{D}\,\mathbf{u}(t)
\end{align*}}
> $$
> where $\mathbf{A}\in\mathbb{R}^{H\times H}$ is the state matrix (how the hidden state evolves on its own), $\mathbf{B}\in\mathbb{R}^{H\times d_{\text{in}}}$ is the input matrix (how each input channel drives the state), $\mathbf{C}\in\mathbb{R}^{d_{\text{out}}\times H}$ is the readout matrix (how the state is projected to each output channel), and $\mathbf{D}\in\mathbb{R}^{d_{\text{out}}\times d_{\text{in}}}$ is the feedforward (skip) matrix that lets the input map directly to the output. The system is still time-invariant; the only change relative to L14a is that the input, output, and hidden-state dimensions are no longer forced to be scalars.

For the S4 construction we will build next, $H$ is tied to the per-channel LegS order $h$ and the number of input channels by $H = h\cdot d_{\text{in}}$. The dimensions then collapse to the single scalar knob we already had in L14a.

> __Dimension Dictionary (MIMO)__
>
> Let $h$ denote the LegS basis order per input channel. Then
> * $\mathbf{u}_{t}\in\mathbb{R}^{d_{\text{in}}}$ is the input vector at time $t$ and $\mathbf{y}_{t}\in\mathbb{R}^{d_{\text{out}}}$ is the output vector at time $t$.
> * $\mathbf{x}_{t}\in\mathbb{R}^{H}$ is the hidden-state vector at time $t$, with total dimension $H = h\cdot d_{\text{in}}$ from running one $h$-dim LegS filter per input channel.
> * $\mathbf{A}\in\mathbb{R}^{H\times H}$ is the (block-diagonal) state matrix that governs how the hidden state evolves on its own, and $\mathbf{B}\in\mathbb{R}^{H\times d_{\text{in}}}$ is the (block-rectangular) input matrix that routes each input channel into its own block of the hidden state.
> * $\mathbf{C}\in\mathbb{R}^{d_{\text{out}}\times H}$ is the (dense) readout matrix that projects the full hidden state to each output channel, and $\mathbf{D}\in\mathbb{R}^{d_{\text{out}}\times d_{\text{in}}}$ is the feedforward matrix that optionally skips the state and maps inputs directly to outputs.
> * SISO is the special case $d_{\text{in}} = d_{\text{out}} = 1$, which recovers the L14a construction exactly.

___

## Block-Diagonal MIMO LegS
The SISO LegS construction in L14a gave a single $h\times h$ state matrix and an $h\times 1$ input column. The cleanest way to extend this to $d_{\text{in}}$ channels without disturbing the HiPPO theory is to run one independent SISO LegS filter per channel and stack their hidden states into a single $H$-dimensional vector with $H = h\cdot d_{\text{in}}$.

> __Block-Diagonal LegS State Matrix__
>
> Let $\mathbf{A}_{\text{LegS}}\in\mathbb{R}^{h\times h}$ and $\mathbf{b}_{\text{LegS}}\in\mathbb{R}^{h}$ denote the single-channel LegS matrices from L14a. The MIMO LegS state matrix and input matrix are
> $$
\boxed{
\mathbf{A} = \mathbf{I}_{d_{\text{in}}}\otimes\mathbf{A}_{\text{LegS}},\qquad
\mathbf{B} = \mathbf{I}_{d_{\text{in}}}\otimes\mathbf{b}_{\text{LegS}}
}
> $$
> where $\otimes$ is the Kronecker product, so $\mathbf{A}$ is block-diagonal with $d_{\text{in}}$ copies of $\mathbf{A}_{\text{LegS}}$ on its diagonal, and column $j$ of $\mathbf{B}$ is $\mathbf{b}_{\text{LegS}}$ placed in block $j$ and zero elsewhere.

The helper [`build_legS_matrices_mimo(h, d_in)`](src/Compute.jl) in `src/Compute.jl` constructs these matrices exactly as above.

Three consequences of block-diagonality are worth pausing on. First, the eigenvalues of $\mathbf{A}$ are the eigenvalues of $\mathbf{A}_{\text{LegS}}$ repeated $d_{\text{in}}$ times, so the stability argument from L14a carries over unchanged. Second, the bilinear-discretized $\bar{\mathbf{A}} = (\mathbf{I} - \tfrac{\Delta t}{2}\mathbf{A})^{-1}(\mathbf{I} + \tfrac{\Delta t}{2}\mathbf{A})$ is also block-diagonal because the inverse of a block-diagonal matrix is block-diagonal. Third, the recursion $\mathbf{x}_{t} = \bar{\mathbf{A}}\,\mathbf{x}_{t-1} + \bar{\mathbf{B}}\,\mathbf{u}_{t}$ decouples across channels: input channel $j$ only drives its own $h$-dimensional block of the hidden state.

The _coupling_ between channels lives entirely in the readout $\mathbf{C}$. Because $\mathbf{C}\in\mathbb{R}^{d_{\text{out}}\times H}$ is dense, every output can take a linear combination of every hidden coordinate across every input channel. That is where the MIMO model earns its expressiveness over a stack of independent SISO observers.

One honest caveat about how this construction maps onto the canonical S4 architecture is worth stating before we move on.

> __Relation to S4 as trained in practice__
>
> The S4 architecture in [Gu, Goel, and Ré (2022)](https://arxiv.org/abs/2111.00396) does not put cross-channel coupling inside a single SSM layer. Instead, S4 runs one independent SISO SSM per input channel with no coupling at the SSM level, and mixes channels in a standard feedforward MLP placed between stacked SSM layers. What we build here places the mixing directly in the readout $\bar{\mathbf{C}}$ of a single SSM layer, which is mathematically valid and pedagogically simpler but architecturally closer to a wide linear observer than to S4's deep SSM-plus-MLP stack. The block-diagonal LegS structure that makes our construction work is exactly the same one S4 uses for each of its per-channel SSMs; the difference is in what we do between, and on top of, those SSMs.

___

## Training: Stacked Ridge Regression on the Shared Hidden State
With $(\bar{\mathbf{A}}, \bar{\mathbf{B}})$ frozen at the LegS initialization, the only learnable matrices are $\bar{\mathbf{C}}$ and (optionally) $\bar{\mathbf{D}}$. The training loss remains quadratic in these matrices, so we still get a closed-form solution; the only change from L14a is that the loss now sums over $d_{\text{out}}$ output channels at every time step.

Before writing down the normal equations, we need to roll out the shared hidden-state matrix exactly once. The roll-out depends only on $\bar{\mathbf{A}}$, $\bar{\mathbf{B}}$, and the input sequence, so it is the same $\mathbf{X}$ that will be used to fit every row of $\bar{\mathbf{C}}$.

> __Stacked Ridge Normal Equations__
>
> Let $\mathbf{U}\in\mathbb{R}^{T\times d_{\text{in}}}$ be a training input (row $t$ is $\mathbf{u}_{t}^{\top}$), let $\mathbf{Y}\in\mathbb{R}^{T\times d_{\text{out}}}$ be the matching target matrix, and let $\mathbf{X}\in\mathbb{R}^{T\times H}$ be the hidden-state rollout of the shared filter on $\mathbf{U}$. With $\bar{\mathbf{D}}$ held fixed (commonly $\bar{\mathbf{D}} = \mathbf{0}$), the ridge-regression loss is
> $$
\mathcal{L}(\bar{\mathbf{C}}) = \sum_{t=1}^{T}\left\|\mathbf{y}_{t} - \bar{\mathbf{C}}\,\mathbf{x}_{t} - \bar{\mathbf{D}}\,\mathbf{u}_{t}\right\|_{2}^{2} + \lambda\,\|\bar{\mathbf{C}}\|_{F}^{2}.
> $$
> This loss decouples across output channels because $\bar{\mathbf{C}}\mathbf{x}_{t}$ is a linear function of each row of $\bar{\mathbf{C}}$ separately. Its minimizer is
> $$
\boxed{\;\;\hat{\bar{\mathbf{C}}}^{\top} = \left(\mathbf{X}^{\top}\mathbf{X} + \lambda\mathbf{I}\right)^{-1}\mathbf{X}^{\top}\,\mathbf{Y}_{\text{res}},\;\;}
> $$
> where $\mathbf{Y}_{\text{res}} = \mathbf{Y} - \mathbf{U}\bar{\mathbf{D}}^{\top}$ removes the feedforward contribution. The same Gram matrix $(\mathbf{X}^{\top}\mathbf{X} + \lambda\mathbf{I})$ is inverted once and reused for every output channel.

A practical consequence is that multi-output MIMO training is essentially free relative to single-output SISO training. The dominant cost is the rollout and the $H\times H$ linear solve; adding output channels only changes the size of the right-hand side from $H\times 1$ to $H\times d_{\text{out}}$, which is negligible for $d_{\text{out}}\ll T$.

The helper [`fit_C!(model, U, Y; λ)`](src/Compute.jl) in `src/Compute.jl` implements these normal equations for a `MyMimoLegSHippoModel`.

___

## From Memorize to Forecast
Up to this point, nothing has pinned down what the target matrix $\mathbf{Y}$ actually represents. In L14a the target equaled the input ($y_{t} = u_{t}$), which made the task a memorization check. A more useful task for time series is _forecasting_: predict a future value of the input from its own past.

> __Forecasting Target__
>
> Given an input sequence $(\mathbf{u}_{1}, \mathbf{u}_{2}, \ldots, \mathbf{u}_{T})$ and a forecast horizon $k\geq 1$, the aligned input and target matrices are
> $$
\boxed{
\mathbf{U}_{\text{input}} = \begin{pmatrix}\mathbf{u}_{1}^{\top}\\ \mathbf{u}_{2}^{\top}\\ \vdots\\ \mathbf{u}_{T-k}^{\top}\end{pmatrix},\qquad
\mathbf{U}_{\text{target}} = \begin{pmatrix}\mathbf{u}_{1+k}^{\top}\\ \mathbf{u}_{2+k}^{\top}\\ \vdots\\ \mathbf{u}_{T}^{\top}\end{pmatrix}
}
> $$
> and training minimizes $\sum_{t=1}^{T-k}\|\mathbf{u}_{t+k} - \bar{\mathbf{C}}\,\mathbf{x}_{t} - \bar{\mathbf{D}}\,\mathbf{u}_{t}\|_{2}^{2}$. A one-step-ahead forecaster uses $k = 1$; longer-horizon forecasters use larger $k$.

The helper [`make_forecast_pair(U, k)`](src/Compute.jl) in `src/Compute.jl` returns exactly this aligned pair.

> __Why forecasting is harder than memorizing__
>
> In the memorize setting the optimal decoder exists analytically: the LegS basis carries the current input in its polynomial-reconstruction coefficients, and the readout only has to extract it. In the forecast setting no such analytical optimum exists in general. The trained readout has to discover, from data, any predictable structure in the signal that survives from time $t$ to time $t+k$. If the signal is white noise, the best possible forecast is the signal's mean, and the correlation of the forecast with the realized future value is essentially zero regardless of how large $h$ grows. If the signal has genuine dynamical structure, the observer can extract it, and the forecast correlation can approach one. The two-channel oscillator example in the companion notebook is in this second regime by construction.

___

## How MIMO SSMs Compare to Classical Time-Series Tools
Linear forecasting of vector time series has a long history outside the SSM literature; two tools dominate applied work. Vector autoregression (VAR) fits a linear model of the current vector on a fixed number of lagged vectors. The Kalman filter and smoother solve optimal state estimation in a linear Gaussian state space model, using a known or jointly-estimated state-space representation. The MIMO SSM sits alongside these as a third, architecturally distinct option.

> __Linear vector-forecasting tools__
>
> | Property | VAR($p$) | Kalman filter | MIMO SSM (S4-LegS) |
> |---|---|---|---|
> | Memory mechanism | Fixed $p$ most recent lags | Learned / assumed state-space dynamics | HiPPO-LegS polynomial summary of full history |
> | Dynamics | Linear, data-fit | Linear, model-specified | Linear, HiPPO-initialized + learned readout |
> | Training cost | OLS on lag-stacked matrix | EM or joint maximum likelihood | One closed-form ridge solve |
> | Memory cost in $T$ | $\mathcal{O}(p\,d)$ per time step | $\mathcal{O}(H\,d)$ per time step | $\mathcal{O}(H\,d)$ per time step |
> | Parameters to learn | $p\,d^{2}$ lag matrices | up to $H^{2} + H\,d$ dynamics entries | $d_{\text{out}}\,H$ entries of the readout |
> | Long-range memory | Truncated at lag $p$ | Geometric decay by state dynamics | Polynomial-approximation guarantee |

The MIMO SSM's distinguishing feature is the separation between a _fixed_, structured memory mechanism (the HiPPO-LegS basis) and a _learned_ linear readout, which leaves only a small ridge-regression problem to solve from data. This is why the parameter count is asymptotically smaller than VAR and Kalman.

> __Pointer to S5__
>
> The S4 family we use today restricts $\mathbf{A}$ to block-diagonal, which is mathematically convenient but gives up some expressiveness relative to an unrestricted MIMO SSM. [Smith, Warrington, and Linderman (2022)](https://arxiv.org/abs/2208.04933) introduce _S5_, which uses a single diagonal $\mathbf{A}$ shared across channels and a dense $\mathbf{B}$ mapping all input channels into every state dimension. S5 is natively MIMO (no block decomposition) and matches or exceeds S4 on the Long Range Arena benchmark with fewer parameters; it is the natural next step after the model we build today.

___

## Applied Example
Everything in this lecture is pulled together in the companion notebook, where we generate a two-channel synthetic signal from a pair of sinusoids at different frequencies, train a MIMO LegS SSM to forecast the signal one step ahead, and evaluate on a held-out continuation of the same series.

> __Example: Two-channel oscillator forecasting__
>
> [▶ Two-channel oscillator forecasting with a MIMO HiPPO-LegS SSM](CHEME-5820-L14c-Example-TwoChannelOscillator-Spring-2026.ipynb). We build a 2-input 2-output MIMO HiPPO-LegS model, train the readout by closed-form ridge regression on $\mathbf{U}_{\text{input}}, \mathbf{U}_{\text{target}}$ at forecast horizon $k=1$, and evaluate out-of-sample on a held-out continuation. We also sweep the hidden dimension $h$ and the horizon $k$ to show how reconstruction quality scales with memory and with prediction lead time.

___

## Summary
A multi-input multi-output structured state space model replaces the scalar input and output of the SISO construction with vectors and extends the HiPPO-LegS basis by running one independent SISO filter per input channel, so the state matrix is block-diagonal and the input matrix is block-rectangular. The readout remains dense and captures all the cross-channel coupling in the system. Because the state and input matrices are still frozen at the HiPPO initialization, the training loss remains quadratic in the readout and can be solved as a single closed-form ridge regression, now stacked across all output channels. Replacing the memorize target with a forecasting target turns the same machinery into a predictor rather than a reconstructor, and exposes whether the input signal actually has predictable structure beyond its present value.

> __Key Takeaways:__
>
> * **Block-diagonal MIMO LegS keeps the SISO guarantees:** The state matrix is block-diagonal with one single-channel LegS block per input channel, so its eigenvalues are the SISO eigenvalues repeated and the bilinear-discretized pair stays stable. The per-channel approximation bounds carry over from the single-channel construction without change.
> * **Multi-output training is one linear solve for all output channels:** With the state and input matrices frozen, the ridge normal equations factor through a single shared Gram matrix and a stacked right-hand side. Adding output channels therefore costs almost nothing beyond the single-output solve.
> * **Forecasting replaces memorize as the honest out-of-sample test:** Aligning the input and target with a positive lead time makes the model predict rather than reconstruct. The forecast correlation on a held-out continuation then reflects whether the input signal carries predictable structure or is effectively white noise.

For an applied example training a 2-input 2-output MIMO HiPPO-LegS model on a two-channel oscillator signal and evaluating forecast quality at multiple horizons, see the [L14c example notebook](CHEME-5820-L14c-Example-TwoChannelOscillator-Spring-2026.ipynb).
___